# hMOF Dataset Splits

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split

PARQUET_PATH = "hmof_dataset_cleaned.parquet"
CO2_COLS = [
    "co2_mol_kg_0.01bar", "co2_mol_kg_0.05bar", "co2_mol_kg_0.1bar",
    "co2_mol_kg_0.5bar",  "co2_mol_kg_2.5bar",
]
RANDOM_STATE = 42

df = pd.read_parquet(PARQUET_PATH)
# df = df.dropna(subset=CO2_COLS + ["topology"])  # safety guard

print(f"Loaded: {len(df):,} rows × {df.shape[1]} columns")
print(f"Topologies: {sorted(df['topology'].unique())}")

In [ ]:
def partition_stats(name: str, split: dict[str, pd.DataFrame]) -> dict:
    """Print per-partition stats and return a summary dict for the comparison table."""
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    row = {"split": name}
    for part, sub in split.items():
        n          = len(sub)
        n_topo     = sub["topology"].nunique()
        n_metal    = sub["metal_node"].nunique()
        co2_mean   = sub[CO2_COLS].mean().mean()
        co2_std    = sub[CO2_COLS].std().mean()
        print(
            f"  {part:6s}  n={n:>7,}  topologies={n_topo:>3}  "
            f"metal_nodes={n_metal:>3}  CO2_mean={co2_mean:.3f}  CO2_std={co2_std:.3f}"
        )
        row[f"{part}_n"]       = n
        row[f"{part}_topo"]    = n_topo
        row[f"{part}_metal"]   = n_metal
        row[f"{part}_co2mean"] = round(co2_mean, 3)
    return row


def assert_no_leakage(train: pd.DataFrame, val: pd.DataFrame,
                      test: pd.DataFrame, col: str) -> None:
    train_vals = set(train[col].dropna())
    assert set(val[col].dropna())  .isdisjoint(train_vals), f"val/{col} leaks into train"
    assert set(test[col].dropna()) .isdisjoint(train_vals), f"test/{col} leaks into train"
    assert set(val[col].dropna())  .isdisjoint(set(test[col].dropna())), f"val/{col} leaks into test"
    print(f"  ✓ zero {col} leakage between train / val / test")

## Split 1 — Random (80 / 10 / 10)

In [ ]:
def make_stratify_col(series: pd.Series, min_count: int = 3) -> pd.Series:
    """
    Return a stratify-safe copy of `series`: classes with fewer than
    `min_count` members are relabelled '__OTHER__' so sklearn can split them.
    The original series is never modified.
    """
    counts  = series.value_counts()
    rare    = counts[counts < min_count].index.tolist()
    strat   = series.copy()
    strat[strat.isin(rare)] = "__OTHER__"
    if rare:
        print(f"  Merged {len(rare)} rare class(es) into '__OTHER__': {rare}")
    else:
        print("  No rare classes — all classes have >= min_count samples.")
    return strat


print("Stratify key: topology")
strat_col = make_stratify_col(df["topology"])

train_rand, tmp = train_test_split(
    df, test_size=0.20, random_state=RANDOM_STATE, stratify=strat_col
)
val_rand, test_rand = train_test_split(
    tmp, test_size=0.50, random_state=RANDOM_STATE,
    stratify=make_stratify_col(tmp["topology"])
)

summary_rand = partition_stats(
    "Random 80/10/10",
    {"train": train_rand, "val": val_rand, "test": test_rand}
)

# Save
train_rand.to_parquet("split_random_train.parquet", index=False)
val_rand  .to_parquet("split_random_val.parquet",   index=False)
test_rand .to_parquet("split_random_test.parquet",  index=False)

## Split 2 — Topology-stratified (unseen topologies in val / test)

In [14]:
def group_split(df: pd.DataFrame, group_col: str,
                val_size: float = 0.10, test_size: float = 0.10,
                random_state: int = RANDOM_STATE) -> tuple[pd.DataFrame, ...]:
    """
    Hold out entire groups (topologies or metal nodes) for val and test.
    Uses GroupShuffleSplit twice: first carve off test groups, then val groups
    from the remaining pool.
    """
    groups = df[group_col].values

    # 1. Split off test groups (~10 % of rows)
    gss_test = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_val_idx, test_idx = next(gss_test.split(df, groups=groups))

    df_trainval = df.iloc[train_val_idx]
    df_test     = df.iloc[test_idx]

    # 2. Split val groups from the train+val pool (~10 % of total ≈ 11 % of remaining)
    val_frac = val_size / (1.0 - test_size)
    groups_tv = df_trainval[group_col].values
    gss_val = GroupShuffleSplit(n_splits=1, test_size=val_frac, random_state=random_state)
    train_idx, val_idx = next(gss_val.split(df_trainval, groups=groups_tv))

    df_train = df_trainval.iloc[train_idx]
    df_val   = df_trainval.iloc[val_idx]

    return df_train, df_val, df_test


train_topo, val_topo, test_topo = group_split(df, group_col="topology")

assert_no_leakage(train_topo, val_topo, test_topo, "topology")

summary_topo = partition_stats(
    "Topology-stratified",
    {"train": train_topo, "val": val_topo, "test": test_topo}
)

# Save
train_topo.to_parquet("split_topology_train.parquet", index=False)
val_topo  .to_parquet("split_topology_val.parquet",   index=False)
test_topo .to_parquet("split_topology_test.parquet",  index=False)

  ✓ zero topology leakage between train / val / test

Topology-stratified
  train   n=  8,809  topologies= 24  metal_nodes= 14  CO2_mean=1.409  CO2_std=1.163
  val     n=  1,048  topologies=  3  metal_nodes= 11  CO2_mean=1.679  CO2_std=0.808
  test    n=101,612  topologies=  3  metal_nodes= 17  CO2_mean=1.585  CO2_std=1.084


## Split 3 — Metal-node-stratified (unseen metal nodes in val / test)

In [15]:
# Rows without a metal_node label are excluded from group-based splitting;
# they are assigned to train to avoid leakage ambiguity.
df_with_metal    = df[df["metal_node"].notna()].copy()
df_no_metal      = df[df["metal_node"].isna()].copy()

train_metal, val_metal, test_metal = group_split(df_with_metal, group_col="metal_node")

# Append unlabelled rows to train
train_metal = pd.concat([train_metal, df_no_metal], ignore_index=True)

assert_no_leakage(train_metal, val_metal, test_metal, "metal_node")

summary_metal = partition_stats(
    "Metal-node-stratified",
    {"train": train_metal, "val": val_metal, "test": test_metal}
)

# Save
train_metal.to_parquet("split_metal_train.parquet", index=False)
val_metal  .to_parquet("split_metal_val.parquet",   index=False)
test_metal .to_parquet("split_metal_test.parquet",  index=False)

  ✓ zero metal_node leakage between train / val / test

Metal-node-stratified
  train   n= 78,241  topologies= 27  metal_nodes= 17  CO2_mean=1.594  CO2_std=1.080
  val     n=  3,994  topologies= 13  metal_nodes=  3  CO2_mean=1.819  CO2_std=1.050
  test    n= 29,234  topologies= 15  metal_nodes=  3  CO2_mean=1.479  CO2_std=1.128


## Summary comparison

In [ ]:
summary = pd.DataFrame([summary_rand, summary_topo, summary_metal]).set_index("split")

# Reorder columns for readability
parts = ["train", "val", "test"]
cols  = [f"{p}_{s}" for p in parts for s in ["n", "topo", "metal", "co2mean"]]
summary = summary[cols]

summary